In [1]:
# import pandas
import pandas as pd

In [2]:
# sample dataset
data = {
    "CustomerID": [101, 102, 103, 104, 105, 106, 103, 108, 109, 110],
    "Name": ["Rahul", "Priya", "Arun", "Sneha", "Kavya", None, "Arun", "Ravi", "Anu", "Vikram"],
    "Age": [25, None, 150, 28, 32, 45, 30, 17, "twenty", 40],
    "City": ["Hyderabad", "Chennai", "Delhi", "Mumbai", "Vijayawada", "Guntur", "Delhi", None, "Hyderabad", "Chennai"],
    "OrderAmount": [1500, 2500, -500, 3200, 4500, 1200, 1800, 900, None, 6000],
    "Email": [
        "rahul@gmail.com",
        "priya@gmail.com",
        "arun@gmail.com",
        "sneha@gmail.com",
        "kavya@gmail.com",
        "invalid-email",
        "arun@gmail.com",
        "ravi@gmail.com",
        None,
        "vikram@gmail.com"
    ]
}

In [3]:
# creating a pandas dataframe
df = pd.DataFrame(data)

In [4]:
df

,CustomerID,Name,Age,City,OrderAmount,Email
0,101,Rahul,25,Hyderabad,1500.0,rahul@gmail.com
1,102,Priya,None,Chennai,2500.0,priya@gmail.com
2,103,Arun,150,Delhi,-500.0,arun@gmail.com
3,104,Sneha,28,Mumbai,3200.0,sneha@gmail.com
4,105,Kavya,32,Vijayawada,4500.0,kavya@gmail.com
5,106,None,45,Guntur,1200.0,invalid-email
6,103,Arun,30,Delhi,1800.0,arun@gmail.com
7,108,Ravi,17,None,900.0,ravi@gmail.com
8,109,Anu,twenty,Hyderabad,NaN,None
9,110,Vikram,40,Chennai,6000.0,vikram@gmail.com


In [5]:
# inspecting data
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10 entries, 0 to 9
Data columns (total 6 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   CustomerID   10 non-null     int64  
 1   Name         9 non-null      object 
 2   Age          9 non-null      object 
 3   City         9 non-null      object 
 4   OrderAmount  9 non-null      float64
 5   Email        9 non-null      object 
dtypes: float64(1), int64(1), object(4)
memory usage: 612.0+ bytes


In [9]:
# checking null values in the required columns
def check_nulls(df, required_columns):
    invalid = df[required_columns].isnull().any(axis=1)

    valid_records = df[~invalid]
    invalid_records = df[invalid].copy()

    invalid_records["Error"] = "Missing required value"

    return valid_records, invalid_records

In [10]:
required_columns = [
    "CustomerID",
    "Name",
    "Age",
    "City",
    "OrderAmount"
]

valid_df, invalid_df = check_nulls(df, required_columns)

In [11]:
# printing
valid_df

,CustomerID,Name,Age,City,OrderAmount,Email
0,101,Rahul,25,Hyderabad,1500.0,rahul@gmail.com
2,103,Arun,150,Delhi,-500.0,arun@gmail.com
3,104,Sneha,28,Mumbai,3200.0,sneha@gmail.com
4,105,Kavya,32,Vijayawada,4500.0,kavya@gmail.com
6,103,Arun,30,Delhi,1800.0,arun@gmail.com
9,110,Vikram,40,Chennai,6000.0,vikram@gmail.com


In [12]:
invalid_df

,CustomerID,Name,Age,City,OrderAmount,Email,Error
1,102,Priya,None,Chennai,2500.0,priya@gmail.com,Missing required value
5,106,None,45,Guntur,1200.0,invalid-email,Missing required value
7,108,Ravi,17,None,900.0,ravi@gmail.com,Missing required value
8,109,Anu,twenty,Hyderabad,NaN,None,Missing required value


In [13]:
# Type validation
df.dtypes

,0
CustomerID,int64
Name,object
Age,object
City,object
OrderAmount,float64
Email,object


In [14]:
#defining expected datatypes
expected_types = {
    "CustomerID": "numeric",
    "Age": "numeric",
    "OrderAmount": "numeric",
    "Name": "string",
    "City": "string",
    "Email": "string"
}

In [15]:
# fxn fot type checking
def check_types(df, expected_types):

    invalid = pd.Series(False, index=df.index)
    errors = pd.Series("", index=df.index)

    for column, expected_type in expected_types.items():

        if expected_type == "numeric":
            check = pd.to_numeric(df[column], errors="coerce").isna()

        elif expected_type == "string":
            check = df[column].notna() & ~df[column].apply(lambda x: isinstance(x, str))

        invalid = invalid | check

        errors.loc[check] = errors.loc[check] + f"{column} has invalid type; "

    valid_records = df[~invalid]
    invalid_records = df[invalid].copy()

    invalid_records["Error"] = errors[invalid]

    return valid_records, invalid_records

In [16]:
valid_type_df, invalid_type_df = check_types(df, expected_types)

In [17]:
print("VALID RECORDS")
display(valid_type_df)

print("INVALID RECORDS")
display(invalid_type_df)

VALID RECORDS


,CustomerID,Name,Age,City,OrderAmount,Email
0,101,Rahul,25,Hyderabad,1500.0,rahul@gmail.com
2,103,Arun,150,Delhi,-500.0,arun@gmail.com
3,104,Sneha,28,Mumbai,3200.0,sneha@gmail.com
4,105,Kavya,32,Vijayawada,4500.0,kavya@gmail.com
5,106,None,45,Guntur,1200.0,invalid-email
6,103,Arun,30,Delhi,1800.0,arun@gmail.com
7,108,Ravi,17,None,900.0,ravi@gmail.com
9,110,Vikram,40,Chennai,6000.0,vikram@gmail.com


INVALID RECORDS


,CustomerID,Name,Age,City,OrderAmount,Email,Error
1,102,Priya,None,Chennai,2500.0,priya@gmail.com,Age has invalid type;
8,109,Anu,twenty,Hyderabad,NaN,None,Age has invalid type; OrderAmount has invalid ...


In [18]:
# range validation
# defining business rules
AGE_MIN = 18
AGE_MAX = 100

ORDER_AMOUNT_MIN = 0

In [19]:
# range validation fxn
def check_range(df):

    invalid = pd.Series(False, index=df.index)
    errors = pd.Series("", index=df.index)

    # Age validation
    age = pd.to_numeric(df["Age"], errors="coerce")

    age_invalid = (age < 18) | (age > 100)

    invalid = invalid | age_invalid
    errors.loc[age_invalid] = errors.loc[age_invalid] + "Age out of range; "

    # OrderAmount validation
    amount = pd.to_numeric(df["OrderAmount"], errors="coerce")

    amount_invalid = amount < 0

    invalid = invalid | amount_invalid
    errors.loc[amount_invalid] = errors.loc[amount_invalid] + "OrderAmount cannot be negative; "

    valid_records = df[~invalid]
    invalid_records = df[invalid].copy()

    invalid_records["Error"] = errors[invalid]

    return valid_records, invalid_records

In [20]:
valid_range_df, invalid_range_df = check_range(df)

In [21]:
print("INVALID RECORDS")
display(invalid_range_df)

INVALID RECORDS


,CustomerID,Name,Age,City,OrderAmount,Email,Error
2,103,Arun,150,Delhi,-500.0,arun@gmail.com,Age out of range; OrderAmount cannot be negati...
7,108,Ravi,17,None,900.0,ravi@gmail.com,Age out of range;


In [22]:
# schema validation
# defining expected schema
expected_schema = {
    "CustomerID": "numeric",
    "Name": "string",
    "Age": "numeric",
    "City": "string",
    "OrderAmount": "numeric",
    "Email": "string"
}

In [23]:
# schema validation schema
def check_schema(df, expected_schema):

    expected_columns = list(expected_schema.keys())
    actual_columns = list(df.columns)

    missing_columns = [
        column for column in expected_columns
        if column not in actual_columns
    ]

    extra_columns = [
        column for column in actual_columns
        if column not in expected_columns
    ]

    if missing_columns or extra_columns:

        print("Schema validation failed!")

        if missing_columns:
            print("Missing columns:", missing_columns)

        if extra_columns:
            print("Unexpected columns:", extra_columns)

        return False

    print("Schema validation passed!")
    return True

In [24]:
check_schema(df, expected_schema)

Schema validation passed!


True

In [25]:
# duplicate check
def check_duplicates(df, key_column):

    duplicate_mask = df[key_column].duplicated(keep=False)

    valid_records = df[~duplicate_mask]
    invalid_records = df[duplicate_mask].copy()

    invalid_records["Error"] = (
        "Duplicate " + key_column
    )

    return valid_records, invalid_records

In [26]:
valid_duplicate_df, invalid_duplicate_df = check_duplicates(
    df,
    "CustomerID"
)

In [27]:
print("VALID RECORDS")
display(valid_duplicate_df)

print("DUPLICATE RECORDS")
display(invalid_duplicate_df)

VALID RECORDS


,CustomerID,Name,Age,City,OrderAmount,Email
0,101,Rahul,25,Hyderabad,1500.0,rahul@gmail.com
1,102,Priya,None,Chennai,2500.0,priya@gmail.com
3,104,Sneha,28,Mumbai,3200.0,sneha@gmail.com
4,105,Kavya,32,Vijayawada,4500.0,kavya@gmail.com
5,106,None,45,Guntur,1200.0,invalid-email
7,108,Ravi,17,None,900.0,ravi@gmail.com
8,109,Anu,twenty,Hyderabad,NaN,None
9,110,Vikram,40,Chennai,6000.0,vikram@gmail.com


DUPLICATE RECORDS


,CustomerID,Name,Age,City,OrderAmount,Email,Error
2,103,Arun,150,Delhi,-500.0,arun@gmail.com,Duplicate CustomerID
6,103,Arun,30,Delhi,1800.0,arun@gmail.com,Duplicate CustomerID


In [28]:
# email validation
def check_email(df):

    invalid = (
        df["Email"].notna() &
        ~df["Email"].str.contains("@", na=False)
    )

    valid_records = df[~invalid]
    invalid_records = df[invalid].copy()

    invalid_records["Error"] = "Invalid email format"

    return valid_records, invalid_records

In [29]:
valid_email_df, invalid_email_df = check_email(df)

display(invalid_email_df)

,CustomerID,Name,Age,City,OrderAmount,Email,Error
5,106,None,45,Guntur,1200.0,invalid-email,Invalid email format


In [30]:
# refrential checking
def check_customer_reference(df, customers):

    valid_customer_ids = set(customers["CustomerID"])

    invalid = ~df["CustomerID"].isin(valid_customer_ids)

    valid_records = df[~invalid]
    invalid_records = df[invalid].copy()

    invalid_records["Error"] = "CustomerID not found in customer master"

    return valid_records, invalid_records

In [32]:
customers = pd.DataFrame({
    "CustomerID": [101, 102, 103, 104, 105, 106, 108, 109, 110]
})

customers

,CustomerID
0,101
1,102
2,103
3,104
4,105
5,106
6,108
7,109
8,110


In [33]:
valid_ref_df, invalid_ref_df = check_customer_reference(
    df,
    customers
)

display(invalid_ref_df)

,CustomerID,Name,Age,City,OrderAmount,Email,Error


In [38]:
# complete validation layer fxn
def validate_data(df):

    result = df.copy()

    # Create validation columns
    result["is_valid"] = True
    result["errors"] = ""

    # 1. NULL CHECK
    required_columns = [
        "CustomerID",
        "Name",
        "Age",
        "City",
        "OrderAmount"
    ]

    for column in required_columns:
        invalid = result[column].isnull()

        result.loc[invalid, "is_valid"] = False
        result.loc[invalid, "errors"] += f"{column} is missing; "


    # 2. TYPE CHECK
    numeric_columns = [
        "CustomerID",
        "Age",
        "OrderAmount"
    ]

    for column in numeric_columns:

        converted = pd.to_numeric(
            result[column],
            errors="coerce"
        )

        invalid = (
            result[column].notna() &
            converted.isna()
        )

        result.loc[invalid, "is_valid"] = False
        result.loc[invalid, "errors"] += (
            f"{column} has invalid type; "
        )


    # 3. RANGE CHECK
    age = pd.to_numeric(
        result["Age"],
        errors="coerce"
    )

    invalid_age = (
        (age < 18) |
        (age > 100)
    )

    result.loc[invalid_age, "is_valid"] = False
    result.loc[invalid_age, "errors"] += "Age out of range; "


    amount = pd.to_numeric(
        result["OrderAmount"],
        errors="coerce"
    )

    invalid_amount = amount < 0

    result.loc[invalid_amount, "is_valid"] = False
    result.loc[invalid_amount, "errors"] += (
        "OrderAmount cannot be negative; "
    )



    # 4. DUPLICATE CHECK
    duplicate = result["CustomerID"].duplicated(
        keep=False
    )

    result.loc[duplicate, "is_valid"] = False
    result.loc[duplicate, "errors"] += (
        "Duplicate CustomerID; "
    )



    # 5. EMAIL CHECK
    invalid_email = (
        result["Email"].notna() &
        ~result["Email"].str.contains(
            "@",
            na=False
        )
    )

    result.loc[invalid_email, "is_valid"] = False
    result.loc[invalid_email, "errors"] += (
        "Invalid email format; "
    )


    # SEPARATE RECORDS
    valid_records = result[
        result["is_valid"]
    ].copy()

    invalid_records = result[
        ~result["is_valid"]
    ].copy()

    return valid_records, invalid_records

In [39]:
valid_df, invalid_df = validate_data(df)

In [40]:
print("VALID RECORDS")
display(valid_df)

VALID RECORDS


,CustomerID,Name,Age,City,OrderAmount,Email,is_valid,errors
0,101,Rahul,25,Hyderabad,1500.0,rahul@gmail.com,True,
3,104,Sneha,28,Mumbai,3200.0,sneha@gmail.com,True,
4,105,Kavya,32,Vijayawada,4500.0,kavya@gmail.com,True,
9,110,Vikram,40,Chennai,6000.0,vikram@gmail.com,True,


In [41]:
print("INVALID RECORDS")
display(invalid_df)

INVALID RECORDS


,CustomerID,Name,Age,City,OrderAmount,Email,is_valid,errors
1,102,Priya,None,Chennai,2500.0,priya@gmail.com,False,Age is missing;
2,103,Arun,150,Delhi,-500.0,arun@gmail.com,False,Age out of range; OrderAmount cannot be negati...
5,106,None,45,Guntur,1200.0,invalid-email,False,Name is missing; Invalid email format;
6,103,Arun,30,Delhi,1800.0,arun@gmail.com,False,Duplicate CustomerID;
7,108,Ravi,17,None,900.0,ravi@gmail.com,False,City is missing; Age out of range;
8,109,Anu,twenty,Hyderabad,NaN,None,False,OrderAmount is missing; Age has invalid type;


In [42]:
print("Valid records:", len(valid_df))
print("Invalid records:", len(invalid_df))

Valid records: 4
Invalid records: 6
